# S03.2 – Честный split, baseline Dummy и метрики

Цель: собрать минимальный end-to-end эксперимент и сравнить с Dummy как нулём здравого смысла.

## Что вы освоите
- Уметь делать корректный train/test split (с `stratify` и `random_state`)
- Понимать, зачем нужен Dummy baseline и какие стратегии бывают
- Считать метрики классификации (accuracy/precision/recall/F1) и матрицу ошибок
- Давать интерпретацию: почему качество выросло/не выросло относительно Dummy

## Важные оговорки
- Мы сознательно используем примитивные модели. Здесь важен протокол и метрики, а не теория моделей.
- Смотрим на test ровно один раз в конце каждого эксперимента.

---


## Среда, воспроизводимость и стиль эксперимента

Перед кодом – несколько правил, которые будут повторяться во всех ноутбуках:

1) **Воспроизводимость**: фиксируем `random_state` / seed.  
2) **Разделение данных**: test‑часть – это *священная зона*. Мы смотрим на неё только в конце.  
3) **Всё, что "обучается" (`.fit`)** должно видеть только train‑данные (иначе легко получить утечки).  
4) **Sanity‑checks**: после каждого шага проверяем, что получился ожидаемый результат (формы, распределения, пересечения и т.д.).


In [1]:
# Импорты: минимальный, но достаточный набор
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Для красивых картинок (простая визуализация)
import matplotlib.pyplot as plt

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


## Общие функции (оценка моделей и печать метрик)

Чтобы не копировать одно и то же вручную, заведём пару функций.

Важно: эти функции *ничего не делают магически*. Мы специально пишем их максимально прозрачно,
чтобы вы видели, какие именно метрики считаются и на каких данных.


In [2]:
def summarize_binary_metrics(y_true, y_pred, *, positive_label=1):
    """Считает базовые метрики бинарной классификации.

    Мы считаем:
    - accuracy: доля верных ответов
    - precision: насколько "чистые" наши позитивные предсказания
    - recall: насколько хорошо мы находим позитивный класс
    - f1: гармоническое среднее precision и recall

    Почему это важно: в задачах безопасности цена FP и FN может быть разной.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
    }

def print_confusion(y_true, y_pred, labels=(0,1)):
    """Печать матрицы ошибок и пояснения, что есть что."""
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    display(df)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} (верно: 0), FP={fp} (ложная тревога), FN={fn} (пропуск), TP={tp} (верно: 1)")
    return cm

def evaluate_model(model, X_train, y_train, X_test, y_test, *, model_name="model"):
    """Обучает модель на train и оценивает на test. Возвращает словарь метрик."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = summarize_binary_metrics(y_test, y_pred)
    print(f"=== {model_name} ===")
    print(metrics)
    print("Confusion matrix:")
    _ = print_confusion(y_test, y_pred)
    return metrics


## Данные

Берём тот же `breast_cancer`. Повторяем загрузку, чтобы каждый ноутбук был самостоятельным.


In [3]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("X shape:", X.shape, "y shape:", y.shape)
print("Positive class meaning:", data.target_names[1])


X shape: (569, 30) y shape: (569,)
Positive class meaning: benign


## Split: train/test + стратификация

Почему `stratify=y` – хорошая привычка:
- если классы несбалансированы, случайный split может случайно "перекосить" долю классов в train/test;
- тогда метрики будут гулять, и сравнение станет нечестным.

Мы делаем split один раз, фиксируем `random_state`.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Class balance (train):")
display(y_train.value_counts(normalize=True))
print("Class balance (test):")
display(y_test.value_counts(normalize=True))

# Sanity-check: в test есть оба класса
assert set(np.unique(y_test)) == {0,1}, "В test должны присутствовать оба класса для корректной оценки"


Train shape: (426, 30) Test shape: (143, 30)
Class balance (train):


,proportion
target,
1,0.626761
0,0.373239


Class balance (test):


,proportion
target,
1,0.629371
0,0.370629


## Baseline: DummyClassifier

Dummy – это "ноль здравого смысла". Если ваша "модель" не лучше Dummy:
- либо признаки не несут сигнал,
- либо протокол/метрика/данные настроены неадекватно,
- либо вы сравниваете разные сплиты и получаете шум.

Мы сравним 2 стратегии:
- `most_frequent`: всегда предсказывает самый частый класс
- `stratified`: предсказывает классы в пропорции обучающей выборки (случайно)


In [5]:
from sklearn.dummy import DummyClassifier

dummy_mf = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy_str = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)

mf_metrics = evaluate_model(dummy_mf, X_train, y_train, X_test, y_test, model_name="Dummy (most_frequent)")
str_metrics = evaluate_model(dummy_str, X_train, y_train, X_test, y_test, model_name="Dummy (stratified)")


=== Dummy (most_frequent) ===
{'accuracy': 0.6293706293706294, 'precision': 0.6293706293706294, 'recall': 1.0, 'f1': 0.7725321888412017}
Confusion matrix:


,pred_0,pred_1
true_0,0,53
true_1,0,90


TN=0 (верно: 0), FP=53 (ложная тревога), FN=0 (пропуск), TP=90 (верно: 1)
=== Dummy (stratified) ===
{'accuracy': 0.5174825174825175, 'precision': 0.6129032258064516, 'recall': 0.6333333333333333, 'f1': 0.6229508196721312}
Confusion matrix:


,pred_0,pred_1
true_0,17,36
true_1,33,57


TN=17 (верно: 0), FP=36 (ложная тревога), FN=33 (пропуск), TP=57 (верно: 1)


## Мини-модель: "дерево-штамп" (stump) как самый простой обучаемый классификатор

Мы *не изучаем* деревья решений как тему. Здесь stump – это:
- объект с интерфейсом `.fit()` / `.predict()`,
- который строит очень простое правило.

Почему это полезно именно сейчас:
- легко сравнивать с Dummy,
- легко интерпретировать: это почти "одно правило".

Важное ограничение: stump может быть слабым, и это нормально – он нужен для тренировки протокола.


In [6]:
from sklearn.tree import DecisionTreeClassifier

stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
stump_metrics = evaluate_model(stump, X_train, y_train, X_test, y_test, model_name="DecisionTree stump (max_depth=1)")


=== DecisionTree stump (max_depth=1) ===
{'accuracy': 0.9230769230769231, 'precision': 0.9157894736842105, 'recall': 0.9666666666666667, 'f1': 0.9405405405405406}
Confusion matrix:


,pred_0,pred_1
true_0,45,8
true_1,3,87


TN=45 (верно: 0), FP=8 (ложная тревога), FN=3 (пропуск), TP=87 (верно: 1)


## Сводное сравнение и интерпретация

Теперь – главный результат S03:
- сравнить Dummy и модель на одном протоколе,
- объяснить, почему качество стало выше/не стало выше.

Мы не делаем "магических" выводов. Только то, что следует из метрик и матрицы ошибок.


In [7]:
summary = pd.DataFrame([
    {"model": "Dummy most_frequent", **mf_metrics},
    {"model": "Dummy stratified", **str_metrics},
    {"model": "DecisionTree stump", **stump_metrics},
]).set_index("model")

display(summary.sort_values("f1", ascending=False))


,accuracy,precision,recall,f1
model,,,,
DecisionTree stump,0.923077,0.915789,0.966667,0.940541
Dummy most_frequent,0.629371,0.629371,1.000000,0.772532
Dummy stratified,0.517483,0.612903,0.633333,0.622951


### Интерпретация (шаблон, который студент должен уметь воспроизвести)

Сформулируйте 3-5 предложений по следующему шаблону:

1) **Baseline**: "Dummy (strategy=...) даёт F1=..., recall=..., значит нулевой уровень такой-то."  
2) **Модель**: "Stump даёт F1=..., recall=..., улучшение/не улучшение относительно Dummy составляет ..."  
3) **Смысл**: "Это означает, что (есть/нет) сигнал в признаках при таком протоколе."  
4) **Ограничение**: "Модель очень простая, поэтому нельзя ожидать максимума качества."  
5) **Следующий шаг** (без действий, просто мысль): "Дальше можно сравнивать другие классификаторы на том же каркасе."
